# Ship the churn model: three runs, one alias, one endpoint, one batch

One row is one telco customer; the pipeline is D01 5.6's, and the threshold is 2.2's 0.278. You log three models to MLflow, register the best one under the alias `champion`, load it back by that alias, save it for a service, then serve it and score a batch from the terminal.

This notebook works inside the activity folder, `D03/05-Evr-Ship-Churn-Model`: MLflow's records (`mlflow.db`, `mlruns/`) and `churn_model.joblib` are written there, beside `app.py`, `score_batch.py`, `customer.json` and `customers.csv`. To start again from scratch, delete `mlflow.db`, `mlruns/` and `churn_model.joblib` in that folder.

## The starter

The cell moves into the activity folder, loads the data, and rebuilds D01 5.6's preparation and split. `MLFLOW_DISABLE_TELEMETRY` stops MLflow from sending usage data.

In [1]:
import os
import warnings
from pathlib import Path

os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
# MLflow 3.16's autolog calls a scikit-learn metric with an argument name that scikit-learn 1.9 renamed,
# and it warns that the input example has integer columns. Both warnings concern the two libraries, not
# your code, so these lines hide them.
warnings.filterwarnings("ignore", message="`y_pred` was renamed")
warnings.filterwarnings("ignore", message="Hint: Inferred schema contains integer")
here = Path.cwd()
if here.name in ("Unsolved", "Solved"):
    os.chdir("..")                                   # the notebook was started in its own folder
elif here.name != "05-Evr-Ship-Churn-Model":
    os.chdir("D03/05-Evr-Ship-Churn-Model")          # VS Code starts notebooks at the repo root
print("working in", Path.cwd().name)

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../../data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)


def make_pre(scale=True):
    """D01 5.6's ColumnTransformer; scale=False for the tree models."""
    num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()) if scale else SimpleImputer(strategy="median")
    return ColumnTransformer([("num", num, num_cols),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])


cv = StratifiedKFold(5, shuffle=True, random_state=42)
THRESHOLD = 0.278
print(len(X_train), "training customers,", len(X_test), "test customers")

working in 05-Evr-Ship-Churn-Model


5634 training customers, 1409 test customers


## Step 1 · Log three runs

Each run fits one model inside `mlflow.start_run(run_name=...)`, logs its 5-fold CV ROC AUC with `mlflow.log_metric("cv_auc", ...)`, and saves the fitted pipeline with `mlflow.sklearn.log_model`. `mlflow.sklearn.autolog(log_models=False)` records every setting for you.

MLflow 3.16 saves scikit-learn models in the skops format, which refuses to load types it was not told to trust. `skops_trusted_types` lists them, so the model can be loaded back in step 4.

In [2]:
import logging

import mlflow
import skops.io as sio
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

logging.getLogger("mlflow").setLevel(logging.ERROR)     # MLflow logs every step it takes; keep only errors
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("telco-churn")
mlflow.sklearn.autolog(log_models=False, silent=True)

candidates = {
    "logistic": make_pipeline(make_pre(), LogisticRegression(max_iter=1000)),
    "forest": make_pipeline(make_pre(scale=False), RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    "boosted": make_pipeline(make_pre(scale=False), HistGradientBoostingClassifier(random_state=42)),
}
model_uris = {}
for name, model in candidates.items():
    with mlflow.start_run(run_name=name):
        cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
        model.fit(X_train, y_train)
        mlflow.log_metric("cv_auc", float(cv_auc.mean()))
        trusted = sio.get_untrusted_types(data=sio.dumps(model))
        info = mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3),
                                        skops_trusted_types=trusted)
        model_uris[name] = info.model_uri
    print(f"{name:<9} cv_auc {cv_auc.mean():.3f}")

logistic  cv_auc 0.846


forest    cv_auc 0.821


boosted   cv_auc 0.838


## Step 2 · Compare the runs

`mlflow.search_runs(order_by=["metrics.cv_auc DESC"])[["tags.mlflow.runName", "metrics.cv_auc"]]`

In [3]:
runs = mlflow.search_runs(order_by=["metrics.cv_auc DESC"])
runs[["tags.mlflow.runName", "metrics.cv_auc"]]

,tags.mlflow.runName,metrics.cv_auc
0,logistic,0.846137
1,boosted,0.837712
2,forest,0.820594


## Step 3 · Register the best run and give it an alias

`mv = mlflow.register_model(uri_of_best, "telco-churn")`, then `MlflowClient().set_registered_model_alias("telco-churn", "champion", mv.version)`. The URI of each run's model is in `model_uris`.

In [4]:
from mlflow import MlflowClient

best = runs.iloc[0]["tags.mlflow.runName"]
mv = mlflow.register_model(model_uris[best], "telco-churn")
MlflowClient().set_registered_model_alias("telco-churn", "champion", mv.version)
print(f"Created version '{mv.version}' of model 'telco-churn' from the {best} run")

Created version '1' of model 'telco-churn' from the logistic run


Successfully registered model 'telco-churn'.
Created version '1' of model 'telco-churn'.


## Step 4 · Load it by alias, the way a service would

`champion = mlflow.sklearn.load_model("models:/telco-churn@champion")`. Score the test customers and print the ROC AUC.

In [5]:
from sklearn.metrics import roc_auc_score

champion = mlflow.sklearn.load_model("models:/telco-churn@champion")
champion_auc = roc_auc_score(y_test, champion.predict_proba(X_test)[:, 1])
print(f"test ROC AUC {champion_auc:.3f}")

test ROC AUC 0.842


## Step 5 · Save the pipeline for the service

`joblib.dump(champion, "churn_model.joblib")`. The file lands in the activity folder, where `app.py` and `score_batch.py` look for it.

In [6]:
import joblib

joblib.dump(champion, "churn_model.joblib")

['churn_model.joblib']

## Steps 6 to 9 · In the terminal

From here on you work in the VS Code terminal, inside the activity folder (`cd D03/05-Evr-Ship-Churn-Model`), as the README describes.

6. Start the endpoint in a second terminal and leave it running: `uvicorn app:app --port 8000`
7. Call it from the first terminal: `curl -X POST localhost:8000/predict -H "Content-Type: application/json" -d @customer.json`
8. Delete the `tenure` line from a copy of `customer.json`, call again, and read the status code and the message.
9. Score a batch: `python score_batch.py customers.csv scores.csv --model churn_model.joblib --threshold 0.278`, then open `scores.csv`.

## Steps 6 to 9 from Python

The same calls, made from this notebook so it can check itself: start the endpoint as a background process, call it with `urllib` (what `curl` does), stop it, then run the batch script.

In [7]:
import json
import subprocess
import sys
import time
import urllib.error
import urllib.request

server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--port", "8000"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def post(payload):
    req = urllib.request.Request("http://localhost:8000/predict", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req) as r:
            return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read())


try:
    for _ in range(60):                                  # wait until the endpoint answers
        try:
            urllib.request.urlopen("http://localhost:8000/health")
            break
        except OSError:
            time.sleep(0.5)
    customer = json.load(open("customer.json"))
    status, answer = post(customer)                      # step 7
    print(status, json.dumps(answer))
    broken = {k: v for k, v in customer.items() if k != "tenure"}
    bad_status, bad_answer = post(broken)                # step 8
    print(bad_status, json.dumps(bad_answer))
finally:
    server.terminate()
    server.wait()

batch = subprocess.run([sys.executable, "score_batch.py", "customers.csv", "scores.csv",
                        "--model", "churn_model.joblib", "--threshold", "0.278"],
                       capture_output=True, text=True, check=True)        # step 9
print(batch.stdout)
pd.read_csv("scores.csv").head()

200 {"churn_probability": 0.045, "flag": false, "threshold": 0.278}
422 {"detail": [{"type": "missing", "loc": ["body", "tenure"], "msg": "Field required", "input": {"gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes", "PhoneService": "Yes", "MultipleLines": "Yes", "InternetService": "Fiber optic", "OnlineSecurity": "Yes", "OnlineBackup": "Yes", "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "Yes", "StreamingMovies": "Yes", "Contract": "Two year", "PaperlessBilling": "Yes", "PaymentMethod": "Credit card (automatic)", "MonthlyCharges": 114.05, "TotalCharges": 8468.2}}]}


scored 1409 customers, flagged 564, wrote scores.csv



,customerID,churn_probability,flag
0,4376-KFVRS,0.045,False
1,2754-SDJRD,0.684,True
2,9917-KWRBE,0.059,False
3,0365-GXEZS,0.402,True
4,9385-NXKDA,0.022,False


## Bonus · A challenger

Register the `class_weight="balanced"` model as version 2 with the alias `challenger`, load it by alias, and compare its test recall at the 0.5 default with the champion's.

In [8]:
from sklearn.metrics import recall_score

with mlflow.start_run(run_name="logistic-balanced"):
    balanced = make_pipeline(make_pre(), LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_train, y_train)
    info2 = mlflow.sklearn.log_model(balanced, name="model",
                                     skops_trusted_types=sio.get_untrusted_types(data=sio.dumps(balanced)))
mv2 = mlflow.register_model(info2.model_uri, "telco-churn")
MlflowClient().set_registered_model_alias("telco-churn", "challenger", mv2.version)
challenger = mlflow.sklearn.load_model("models:/telco-churn@challenger")
recall_champion = recall_score(y_test, champion.predict(X_test))
recall_challenger = recall_score(y_test, challenger.predict(X_test))
print(f"recall: champion {recall_champion:.3f}, challenger {recall_challenger:.3f}")

Registered model 'telco-churn' already exists. Creating a new version of this model...
Created version '2' of model 'telco-churn'.


recall: champion 0.559, challenger 0.783


## Check yourself

In [9]:
scored = pd.read_csv("scores.csv")
check = {
    "Step 2 · runs": "; ".join(f"{r['tags.mlflow.runName']} cv_auc {r['metrics.cv_auc']:.3f}" for _, r in runs.iterrows()),
    "Step 3 · registered version": mv.version,
    "Step 4 · champion test ROC AUC": round(champion_auc, 3),
    "Step 7 · customer": df.loc[X_test.index[0], "customerID"],
    "Step 7 · response": json.dumps(answer),
    "Step 8 · status": bad_status,
    "Step 8 · error": (f"type {bad_answer['detail'][0]['type']}, loc {bad_answer['detail'][0]['loc']}, "
                       f"msg {bad_answer['detail'][0]['msg']}"),
    "Step 9 · customers scored": len(scored),
    "Step 9 · customers flagged": int(scored["flag"].sum()),
    "Step 9 · columns": ", ".join(scored.columns),
    "Bonus · challenger version": mv2.version,
    "Bonus · champion recall": round(recall_champion, 3),
    "Bonus · challenger recall": round(recall_challenger, 3),
}
for label, value in check.items():
    print(f"{label:<32} {value}")

Step 2 · runs                    logistic cv_auc 0.846; boosted cv_auc 0.838; forest cv_auc 0.821
Step 3 · registered version      1
Step 4 · champion test ROC AUC   0.842
Step 7 · customer                4376-KFVRS
Step 7 · response                {"churn_probability": 0.045, "flag": false, "threshold": 0.278}
Step 8 · status                  422
Step 8 · error                   type missing, loc ['body', 'tenure'], msg Field required
Step 9 · customers scored        1409
Step 9 · customers flagged       564
Step 9 · columns                 customerID, churn_probability, flag
Bonus · challenger version       2
Bonus · champion recall          0.559
Bonus · challenger recall        0.783
